# valley-axis demo

This notebook runs the full `valley_axis` pipeline on the included sample data and visualises the three outputs:
- **Centerlines** (vector GeoDataFrame)
- **Centerlines raster**
- **Valley width raster** (IDW-interpolated)


In [2]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rioxarray

from valley_axis import derive_axis
from valley_axis.sample_data import get_sample_data

## Load sample data

In [5]:
data = get_sample_data()

dem = rioxarray.open_rasterio(data["dem"]).squeeze()
region = rioxarray.open_rasterio(data["region"]).squeeze()
flowlines = gpd.read_file(data["flowlines"])

print(f"DEM shape: {dem.shape}, CRS: {dem.rio.crs}")
print(f"Region shape: {region.shape}")
print(f"Flowlines: {len(flowlines)} features")

DEM shape: (4693, 4401), CRS: EPSG:3310
Region shape: (4693, 4401)
Flowlines: 248 features


## Run pipeline

In [ ]:
centerlines_gdf, centerlines_raster, widths_raster = derive_axis(dem, region, flowlines)

print(f"Centerline segments: {len(centerlines_gdf)}")
print(f"Centerlines raster — non-zero pixels: {(centerlines_raster.values > 0).sum()}")
print(f"Widths raster — valid pixels: {(~__import__('numpy').isnan(widths_raster)).sum()}")

## Visualise

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# DEM + centerlines overlay
ax = axes[0]
dem.plot(ax=ax, cmap="terrain", add_colorbar=True, cbar_kwargs={"label": "Elevation (m)"})
centerlines_gdf.plot(ax=ax, color="red", linewidth=1)
ax.set_title("DEM + Centerlines")
ax.set_aspect("equal")

# Valley mask + centerlines raster
ax = axes[1]
region.plot(ax=ax, cmap="Greens", add_colorbar=False, alpha=0.4)
cl_vals = centerlines_raster.values.astype(float)
cl_vals[cl_vals == 0] = np.nan
axes[1].imshow(
    cl_vals,
    extent=[
        float(dem.x.min()), float(dem.x.max()),
        float(dem.y.min()), float(dem.y.max())
    ],
    origin="upper",
    cmap="Reds",
    interpolation="none",
)
ax.set_title("Valley Mask + Centerlines Raster")
ax.set_aspect("equal")

# Width raster
ax = axes[2]
im = ax.imshow(
    widths_raster,
    extent=[
        float(dem.x.min()), float(dem.x.max()),
        float(dem.y.min()), float(dem.y.max())
    ],
    origin="upper",
    cmap="viridis",
)
plt.colorbar(im, ax=ax, label="Width (m)")
ax.set_title("Valley Width Raster")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()